In [1]:
import Pkg
Pkg.activate("..")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D`


In [2]:
using JLD2
using StaticArrays
using WaveGreen2D: setwave!, wave
using WaveGreen2D.Chebyshev: ChebyshevSeries, gradient, hessian, contains, reduce

In [3]:
setwave!(depth=1.0, wavenumber=1.0)

Wave parameters: h = 1.0 m, ω = 2.73 rad/s, k₀ = 1.0 m, g = 9.81 m/s²

In [4]:
# cs_file = joinpath(@__DIR__, "chebyshev_series.jld2")
cs_file = joinpath("../src/chebyshev_series.jld2")
cs_jld2 = jldopen(cs_file)

const L₁_series = read(cs_jld2, "L₁_series")
const L₂_series = read(cs_jld2, "L₂_series")

close(cs_jld2)

In [5]:
mutable struct ReducedChebyshevSeries
    L₁::ChebyshevSeries{Float64, 2}
    L₂::ChebyshevSeries{Float64, 2}
end

# Reduced series initizalizer
const integrals = ReducedChebyshevSeries(
    ChebyshevSeries(
        Array{Float64, 2}(undef, 1, 1),
        zero(SVector{2, Float64}),
        zero(SVector{2, Float64})
    ),
    ChebyshevSeries(
        Array{Float64, 2}(undef, 1, 1),
        zero(SVector{2, Float64}),
        zero(SVector{2, Float64})
    ),
);

In [6]:
function setintegrals!(h::Real, ω::Real, g::Real)
    H = Float64(h * ω^2/g)
    H̃ = log(H)

    x = SVector{3, Float64}(0.0, 0.0, H)
    x̃ = SVector{3, Float64}(0.0, 0.0, H̃)

    if contains(L₁_series[1], x)
        integrals.L₁ = reduce(L₁_series[1], H; dim=3)
    elseif contains(L₁_series[2], x̃)
        integrals.L₁ = reduce(L₁_series[2], H̃; dim=3)
    elseif contains(L₁_series[3], x̃)
        integrals.L₁ = reduce(L₁_series[3], H̃; dim=3)
    else
        @warn """Green function and its derivatives might be innacurate in the near field \
                 domain due to the very low value of depth compared to the wavelength"""
        integrals.L₁ = reduce(L₁_series[1], H; dim=3)
    end

    if contains(L₂_series[1], x̃)
        integrals.L₂ = reduce(L₂_series[1], H̃; dim=3)
    elseif contains(L₂_series[2], x̃)
        integrals.L₂ = reduce(L₂_series[2], H̃; dim=3)
    elseif contains(L₂_series[3], x̃)
        integrals.L₂ = reduce(L₂_series[3], H̃; dim=3)
    elseif contains(L₂_series[4], x̃)
        integrals.L₂ = reduce(L₂_series[4], H̃; dim=3)
    else
        @warn """Green function and its derivatives might be innacurate in the near field \
                 domain due to the very low value of depth compared to the wavelength"""
        integrals.L₂ = reduce(L₂_series[1], H̃; dim=3)
    end
    
    return nothing
end;

In [7]:
integrals.L₁

2-dimensional Chebyshev series of order (0, 0) for x ∈ [0.0, 0.0]×[0.0, 0.0]

In [8]:
integrals.L₂

2-dimensional Chebyshev series of order (0, 0) for x ∈ [0.0, 0.0]×[0.0, 0.0]

In [10]:
setintegrals!(wave.h, wave.ω, wave.g)

In [11]:
integrals.L₁

2-dimensional Chebyshev series of order (14, 19) for x ∈ [0.0, 0.5]×[0.0, 1.0]

In [12]:
integrals.L₂

2-dimensional Chebyshev series of order (11, 18) for x ∈ [0.0, 0.5]×[0.0, 2.0]

In [13]:
function Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h

    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h

    u = SVector{2, Float64}(A, B₁)

    return integrals.L₁(u)
end


function ∇Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h
    dA_dx = sign(R̄)/wave.h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h
    dB₁_dz = sign(v̄₁)/wave.h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)

    L, ∇ᵤL = gradient(integrals.L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u
    
    return L, ∇L
end


function HΛ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h
    dA_dx = sign(R̄)/wave.h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h
    dB₁_dz = sign(v̄₁)/wave.h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)
    ∇uᵈ = SMatrix{2, 2, Float64}([dA_dx 0.0; 0.0 dB₁_dz])

    L, ∇ᵤL, HᵤL = hessian(integrals.L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u

    # ∂²L/∂x² = ∂²L/∂u² ⋅ (∂u/∂x)²
    HL = ∇uᵈ * HᵤL * ∇uᵈ
    
    return L, ∇L, HL
end;

In [19]:
P = SA[1.0, -2.0]
Q = SA[1.2, -1.0];

In [20]:
wave.h * wave.ω^2 / wave.g

0.7615941559557649

In [21]:
wave.h

1.0

In [22]:
Λ₁(P, Q)

-0.8569054843639957

In [23]:
∇Λ₁(P, Q)

(-0.8569054843639957, [-0.2393810736887632, 0.2939891194753841])

In [24]:
HΛ₁(P, Q)

(-0.8569054843639957, [-0.2393810736887632, 0.2939891194753841], [-1.0644263342079108 -0.698594126559414; -0.698594126559414 1.0644263342215734])

In [25]:
L₁ = L₁_series[1]

3-dimensional Chebyshev series of order (14, 19, 19) for x ∈ [0.0, 0.5]×[0.0, 1.0]×[0.01, 1.64]

In [26]:
H = 0.3
L₁_2D = reduce(L₁, H; dim=3)

B = 0.5
L₁_1D = reduce(L₁_2D, B; dim=2)

A = 0.1
L₁_0D = reduce(L₁_1D, A; dim=1)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

0.0

In [27]:
B = 0.5
L₁_2D = reduce(L₁, B; dim=2)

H = 0.3
L₁_1D = reduce(L₁_2D, H; dim=2)

A = 0.1
L₁_0D = reduce(L₁_1D, A; dim=1)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

0.0

In [28]:
A = 0.1
L₁_2D = reduce(L₁, A; dim=1)

B = 0.5
L₁_1D = reduce(L₁_2D, B; dim=1)

H = 0.3
L₁_0D = reduce(L₁_1D, H)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

2.220446049250313e-16

In [29]:
@code_warntype reduce(L₁_2D, B; dim=1)

MethodInstance for Core.kwcall(::@NamedTuple{dim::Int64}, ::typeof(reduce), ::ChebyshevSeries{Float64, 2}, ::Float64)
  from kwcall(::NamedTuple, ::typeof(reduce), f::ChebyshevSeries{T, N}, x::T) where {T, N} @ WaveGreen2D.Chebyshev ~/repos/WaveGreen2D/src/Chebyshev/clenshaw.jl:61
Static Parameters
  T = Float64
  N = 2
Arguments
  _::Core.Const(Core.kwcall)
  @_2::@NamedTuple{dim::Int64}
  @_3::Core.Const(WaveGreen2D.Chebyshev.reduce)
  f::ChebyshevSeries{Float64, 2}
  x::Float64
Locals
  dim::Union{}
  @_7::Int64
Body::ChebyshevSeries{Float64, 1}
1 ──       Core.NewvarNode(:(dim))
│          Core.NewvarNode(:(@_7))
│    %3  = Core.isdefined(@_2, :dim)::Core.Const(true)
└───       goto #6 if not %3
2 ── %5  = Core.getfield(@_2, :dim)::Int64
│    %6  = WaveGreen2D.Chebyshev.Int::Core.Const(Int64)
│    %7  = (%5 isa %6)::Core.Const(true)
└───       goto #4 if not %7
3 ──       goto #5
4 ──       Core.Const(:(WaveGreen2D.Chebyshev.Int))
│          Core.Const(:(%new(Core.TypeError, Symbol